# Phase 2: Tool-Calling Trajectory Generation

Generate training data by running the **GPT-5.4 teacher agent** on ~200 diverse tickers.

**Pipeline:**
1. Fire off all tickers concurrently via `tqdm_asyncio.gather` (with semaphore for concurrency control)
2. Each task retries up to 5x with exponential backoff — failures don't crash the batch
3. Save **raw trajectories** to `data/bbb/trajectories_raw.jsonl`
4. Convert to **Hermes/chat format** for SFT (truncate tool outputs, add `<think>` tags)
5. Quality filter and save to `data/bbb/trajectories_sft.jsonl`

## Setup

In [1]:
import asyncio
import json
import os
import random
import sys
from pathlib import Path

from openai import AsyncOpenAI
from dotenv import load_dotenv
from tqdm.asyncio import tqdm_asyncio
from tenacity import retry, stop_after_attempt, wait_exponential

# Add nb/ to path so we can import bbb as a package
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "nb"))

from bbb.tools import TOOL_SCHEMAS, TOOL_FUNCTIONS
from bbb.agent import run_tool_calling_agent
from bbb.helpers__data_gen import (
    SYSTEM_PROMPT,
    TICKERS,
    FOCUS_AREAS,
    make_user_prompt,
    serialize_input_list,
    responses_to_hermes,
    filter_trajectory,
    check_tool_errors,
    trajectory_token_stats,
    count_tokens,
)

load_dotenv(PROJECT_ROOT / ".env")

client = AsyncOpenAI(base_url="https://us.api.openai.com/v1")

MODEL = "gpt-5.4"

# Output paths
DATA_DIR = PROJECT_ROOT / "data" / "bbb"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH = DATA_DIR / "trajectories_raw.jsonl"
SFT_PATH = DATA_DIR / "trajectories_sft.jsonl"
SKIPPED_PATH = DATA_DIR / "trajectories_skipped.jsonl"

print(f"Tickers: {len(TICKERS)}")
print(f"Raw output: {RAW_PATH}")
print(f"SFT output: {SFT_PATH}")
print(f"Skipped log: {SKIPPED_PATH}")

Tickers: 191
Raw output: /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/trajectories_raw.jsonl
SFT output: /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/trajectories_sft.jsonl
Skipped log: /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/trajectories_skipped.jsonl


## Configuration

- `MAX_TICKERS`: set to a small number for testing, `None` for the full run
- `FOCUSES_PER_TICKER`: how many focus areas per ticker (max 8). 5 × 191 tickers = 955 trajectories
- `MAX_CONCURRENT`: semaphore limit — controls how many agent loops run in parallel
- Retries: 5 attempts with exponential backoff (4s → 8s → 16s → 32s → 60s)

In [2]:
# Set to a small number (e.g. 5) for testing, None for the full run
MAX_TICKERS = None

# How many different focus areas per ticker (max 8, there are 8 total)
FOCUSES_PER_TICKER = 5

# Max concurrent agent tasks (controls yfinance + API pressure)
MAX_CONCURRENT = 30

# Teacher model reasoning effort
REASONING_EFFORT = "medium"

# Shuffle tickers for diversity in partial runs
tickers = list(TICKERS)
random.shuffle(tickers)
if MAX_TICKERS is not None:
    tickers = tickers[:MAX_TICKERS]

# Build (ticker, focus) pairs — 5 random focuses per ticker
task_pairs = []
for t in tickers:
    focuses = random.sample(FOCUS_AREAS, min(FOCUSES_PER_TICKER, len(FOCUS_AREAS)))
    for f in focuses:
        task_pairs.append((t, f))

print(f"Tickers: {len(tickers)} × {FOCUSES_PER_TICKER} focuses = {len(task_pairs)} trajectories")
print(f"Max concurrent: {MAX_CONCURRENT}")
print(f"Sample pairs: {task_pairs[:5]}")

Tickers: 191 × 5 focuses = 955 trajectories
Max concurrent: 30
Sample pairs: [('DHR', 'risk factors and key challenges ahead'), ('DHR', 'cash flow generation and capital allocation strategy'), ('DHR', 'financial health and balance sheet strength'), ('DHR', 'growth potential and market expansion opportunities'), ('DHR', 'competitive position and market share dynamics')]


## Generate Raw Trajectories

All tickers are dispatched concurrently (bounded by semaphore). Each task:
1. Runs the teacher agent (async OpenAI calls, yfinance in thread pool)
2. Retries up to 5x on failure with exponential backoff
3. On final failure, returns an error dict — does NOT crash the batch

In [3]:
# Skip already-completed (ticker, focus) pairs (for resuming interrupted runs)
completed_pairs = set()
if RAW_PATH.exists():
    with open(RAW_PATH) as f:
        for line in f:
            rec = json.loads(line)
            completed_pairs.add((rec["ticker"], rec.get("focus", "")))
    print(f"Resuming — {len(completed_pairs)} (ticker, focus) pairs already completed")

remaining = [(t, f) for t, f in task_pairs if (t, f) not in completed_pairs]
print(f"Remaining: {len(remaining)} trajectories to generate")

Remaining: 955 trajectories to generate


In [5]:
semaphore = asyncio.Semaphore(MAX_CONCURRENT)
_write_lock = asyncio.Lock()
_skip_lock = asyncio.Lock()

# Counters for summary
_saved = 0
_skipped = 0


@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=2, min=4, max=60),
    reraise=True,
)
async def _run_with_retry(ticker: str, focus: str) -> dict:
    """Run the teacher agent with retries. Raises on failure."""
    async with semaphore:
        return await run_tool_calling_agent(
            client=client,
            model=MODEL,
            user_prompt=make_user_prompt(ticker, focus),
            system_prompt=SYSTEM_PROMPT,
            reasoning_effort=REASONING_EFFORT,
            verbose=False,
        )


async def generate_one(ticker: str, focus: str) -> dict:
    """Generate a single trajectory with quality checks before saving."""
    global _saved, _skipped
    try:
        result = await _run_with_retry(ticker, focus)

        record = {
            "ticker": ticker,
            "focus": focus,
            "input": serialize_input_list(result["input"]),
            "output": serialize_input_list(result["output"]),
            "tools": TOOL_SCHEMAS,
            "reasoning_summaries": result["reasoning_summaries"],
            "usage": result["usage"],
        }

        # --- Quality gate: check BEFORE saving ---
        passes, reason = filter_trajectory(record)
        tool_health = check_tool_errors(record)

        if not passes:
            _skipped += 1
            async with _skip_lock:
                with open(SKIPPED_PATH, "a") as f:
                    f.write(json.dumps({
                        "ticker": ticker,
                        "focus": focus,
                        "reason": reason,
                        "tool_errors": tool_health["errors"],
                    }) + "\n")
            return {"ticker": ticker, "focus": focus, "skipped": reason}

        # Log tool errors as warnings (trajectory still saved if under threshold)
        if tool_health["errored_outputs"] > 0:
            for name, kw, snippet in tool_health["errors"]:
                print(f"  [WARN] {ticker}: {name}() → {kw}")

        # Save to raw file
        async with _write_lock:
            with open(RAW_PATH, "a") as f:
                f.write(json.dumps(record) + "\n")

        _saved += 1
        return record

    except Exception as e:
        _skipped += 1
        async with _skip_lock:
            with open(SKIPPED_PATH, "a") as f:
                f.write(json.dumps({
                    "ticker": ticker,
                    "focus": focus,
                    "reason": f"exception: {str(e)[:200]}",
                }) + "\n")
        return {"ticker": ticker, "focus": focus, "error": str(e)}

In [6]:
tasks = [generate_one(t, f) for t, f in remaining]
results = await tqdm_asyncio.gather(*tasks, desc="Generating trajectories")

# Summarize
successes = [r for r in results if "error" not in r and "skipped" not in r]
skipped = [r for r in results if "skipped" in r]
errors = [r for r in results if "error" in r]

print(f"\nDone: {len(successes)} saved, {len(skipped)} skipped, {len(errors)} failed")
if skipped:
    print(f"\nSkipped (quality filter):")
    for s in skipped[:20]:
        print(f"  {s['ticker']} ({s['focus'][:30]}...): {s['skipped']}")
if errors:
    print(f"\nFailed (exceptions):")
    for e in errors[:20]:
        print(f"  {e['ticker']} ({e['focus'][:30]}...): {e['error'][:80]}")
    if len(errors) > 20:
        print(f"  ... and {len(errors) - 20} more")

print(f"\nSkipped log: {SKIPPED_PATH}")

Generating trajectories:   0%|          | 0/955 [00:00<?, ?it/s]HTTP Error 404: 
$K: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
Generating trajectories:   7%|▋         | 70/955 [02:51<10:18,  1.43it/s]  HTTP Error 404: 
$IPG: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
HTTP Error 404: 
HTTP Error 404: 
$IPG: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
Generating trajectories:   7%|▋         | 71/955 [02:57<27:42,  1.88s/it]HTTP Error 404: 
HTTP Error 404: 
$IPG: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")
Generating trajectories:   8%|▊         | 76/955 [03:08<28:12,  1.93s/it]$IPG: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
HTTP Error 404: 
HTTP Error 404: 
Generating


Done: 955 saved, 0 skipped, 0 failed

Skipped log: /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/trajectories_skipped.jsonl


## Inspect Raw Trajectories

In [7]:
# Load all raw trajectories
raw_records = []
with open(RAW_PATH) as f:
    for line in f:
        raw_records.append(json.loads(line))

print(f"Total raw trajectories: {len(raw_records)}")

# Stats
tool_counts = []
tool_output_lengths = []
memo_lengths = []

for rec in raw_records:
    all_items = rec["input"] + rec["output"]

    calls = [it for it in all_items if isinstance(it, dict) and it.get("type") == "function_call"]
    tool_counts.append(len(calls))

    for it in all_items:
        if isinstance(it, dict) and it.get("type") == "function_call_output":
            tool_output_lengths.append(len(it["output"]))

    for it in reversed(all_items):
        if isinstance(it, dict) and it.get("type") == "message":
            for c in it.get("content", []):
                if isinstance(c, dict) and c.get("text"):
                    memo_lengths.append(len(c["text"]))
                    break
            break

if tool_counts:
    print(f"Tool calls per trajectory: min={min(tool_counts)}, max={max(tool_counts)}, avg={sum(tool_counts)/len(tool_counts):.1f}")
if tool_output_lengths:
    print(f"Tool output chars: min={min(tool_output_lengths)}, max={max(tool_output_lengths)}, avg={sum(tool_output_lengths)/len(tool_output_lengths):.0f}")
if memo_lengths:
    print(f"Memo chars: min={min(memo_lengths)}, max={max(memo_lengths)}, avg={sum(memo_lengths)/len(memo_lengths):.0f}")

Total raw trajectories: 955
Tool calls per trajectory: min=4, max=19, avg=6.3
Tool output chars: min=74, max=43348, avg=9517
Memo chars: min=2516, max=4194, avg=3201


## Convert to SFT Format (Hermes/Chat)

- Reasoning summaries → `<think>` tags
- Tool outputs truncated to ~600 tokens
- Flat schemas → nested Chat Completions format
- Quality filter: must have tool calls, valid JSON args, final output, tool diversity

In [8]:
sft_records = []
filtered_out = []

for rec in raw_records:
    passes, reason = filter_trajectory(rec)
    if not passes:
        filtered_out.append((rec["ticker"], reason))
        continue

    hermes = responses_to_hermes(rec, max_tool_tokens=250)
    if hermes is None:
        filtered_out.append((rec["ticker"], "conversion_failed"))
        continue

    hermes["ticker"] = rec["ticker"]
    hermes["focus"] = rec.get("focus", "")
    sft_records.append(hermes)

print(f"Converted: {len(sft_records)} trajectories")
print(f"Filtered out: {len(filtered_out)}")
if filtered_out:
    for t, r in filtered_out:
        print(f"  {t}: {r}")

Converted: 955 trajectories
Filtered out: 0


In [9]:
with open(SFT_PATH, "w") as f:
    for rec in sft_records:
        f.write(json.dumps(rec) + "\n")

print(f"Saved {len(sft_records)} SFT trajectories to {SFT_PATH}")

Saved 955 SFT trajectories to /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/trajectories_sft.jsonl


## Inspect a Converted Trajectory

In [10]:
if sft_records:
    sample = sft_records[0]
    print(f"Ticker: {sample['ticker']} — Focus: {sample['focus']}")
    print(f"Messages: {len(sample['messages'])} | Tools: {len(sample['tools'])}\n")

    for i, msg in enumerate(sample["messages"]):
        role = msg["role"].upper()
        content = msg.get("content", "")
        tc = msg.get("tool_calls", [])

        if role == "SYSTEM":
            print(f"[{i}] {role}: {content[:80]}...")
        elif role == "USER":
            print(f"[{i}] {role}: {content}")
        elif role == "ASSISTANT" and tc:
            think = "<think>" in (content or "")
            print(f"[{i}] {role} (think={think}, {len(tc)} tool calls): {[t['function']['name'] for t in tc]}")
        elif role == "ASSISTANT":
            print(f"[{i}] {role} (final): {len(content or '')} chars")
        elif role == "TOOL":
            print(f"[{i}] {role}: {len(content or '')} chars")
        print()

Ticker: SHW — Focus: recent news and analyst sentiment
Messages: 9 | Tools: 4

[0] SYSTEM: You are a sell-side equity research analyst producing a brief research snapshot....

[1] USER: Research SHW focusing on recent news and analyst sentiment.

[2] ASSISTANT (think=True, 5 tool calls): ['get_stock_news', 'get_recommendations', 'get_price_history', 'get_financials', 'get_financials']

[3] TOOL: 1084 chars

[4] TOOL: 686 chars

[5] TOOL: 615 chars

[6] TOOL: 777 chars

[7] TOOL: 785 chars

[8] ASSISTANT (final): 5869 chars



## Dataset Statistics

Token counts per role — verifies data fits within `max_seq_length=8192` for SFT.

In [11]:
if sft_records:
    all_stats = [trajectory_token_stats(rec) for rec in sft_records]

    for role in ["system", "user", "assistant", "tool", "total"]:
        vals = [s[role] for s in all_stats]
        print(f"{role:>10}: avg={sum(vals)/len(vals):.0f}  min={min(vals)}  max={max(vals)}")

    over_limit = sum(1 for s in all_stats if s["total"] > 8192)
    print(f"\nTrajectories > 8192 tokens: {over_limit}/{len(all_stats)}")

    system: avg=318  min=318  max=318
      user: avg=12  min=10  max=14
 assistant: avg=2330  min=1010  max=5241
      tool: avg=1587  min=382  max=4851
     total: avg=4247  min=1756  max=9344

Trajectories > 8192 tokens: 13/955
